# Transfer Learning: Borrowing Features Instead of Learning Them

Training a CNN from scratch on CIFAR-10 gets you a mediocre classifier, because 50,000 small images
are not enough to learn good visual features from nothing. But somebody has already trained networks
on ImageNet — 1.2 million photographs across 1,000 categories — and the edge, texture, and part
detectors those models learned are not specific to ImageNet's classes. They are just what useful
visual features look like.

**Transfer learning** takes those features and reuses them. We load a pretrained network, throw away
its classification layer, freeze everything that remains, and train only a small new head on our own
classes. The result beats a from-scratch CNN while training in a fraction of the time, on a fraction
of the data.

## Learning objectives

- Load a pretrained ImageNet model with `include_top=False` and explain what that discards.
- Freeze a base model with `.trainable = False` and say which weights the optimizer can then touch.
- Resize a dataset to match a pretrained model's expected input size.
- Precompute frozen features once, and explain why that makes head training dramatically faster.
- Reassemble the frozen base and the trained head into a single end-to-end model.

## Background

You should be comfortable with the Keras functional API from `U2-2_CNN-5_Multimodal.ipynb`, and with
the from-scratch CIFAR-10 CNN in `U2-2_CNN-4_Cifar.ipynb` — that model's accuracy is the number this
notebook is trying to beat.

Two ideas carry the notebook. First, a trained CNN is really **two parts**: a convolutional stack
that turns pixels into a feature vector, and a small classifier that turns that vector into a label.
Only the second part is specific to the original task, so `include_top=False` keeps the reusable
half and discards the ImageNet-specific one.

Second, **freezing**. Setting `base_model.trainable = False` marks its weights as non-trainable, so
gradient descent leaves them alone. That matters for more than speed: our head starts with random
weights and would emit large, meaningless gradients in the first few batches, which would damage the
very features we came here to borrow.

## This notebook covers

1. Loading a balanced CIFAR-10 subset and resizing it to 224×224
2. Loading the frozen pretrained base
3. Extracting features once, up front
4. Training a small classification head on those features
5. Stitching the base and head back into one end-to-end model
6. Review

**Prerequisites:** `U2-2_CNN-4_Cifar.ipynb` for the from-scratch baseline;
`U2-2_CNN-5_Multimodal.ipynb` for the functional API.

**Dataset:** CIFAR-10, loaded via `tensorflow.keras.datasets.cifar10` and subsampled to 1,000
training and 200 test images per class.

**References:** https://keras.io/api/applications/ and https://keras.io/guides/transfer_learning/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. Load and resize the images

### 1.1 Load a balanced CIFAR-10 subset

Transfer learning's main selling point is that it works with *little* data, so we deliberately use a
small slice: 1,000 training and 200 test images per class rather than the full 50,000/10,000. The
`sample` helper draws an equal number from each class, keeping the subset balanced.

This also keeps the notebook runnable in class — the resizing in section 1.2 is memory-hungry.

In [ ]:
from tensorflow.keras.datasets import cifar10

# Take an EQUAL number of samples from each of the 10 classes, so the subset stays balanced.
def sample(data, labels, num_samples_per_digit):
    sampled_data = []
    sampled_labels = []
    for digit in range(10):
        digit_indices = np.where(labels == digit)[0]
        sampled_indices = np.random.choice(digit_indices, num_samples_per_digit, replace=False)
        sampled_data.append(data[sampled_indices])
        sampled_labels.append(labels[sampled_indices])
    return np.concatenate(sampled_data), np.concatenate(sampled_labels)

# Load data
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# CIFAR-10's classes, in label order (0 = airplane, 1 = automobile, ...).
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

# Define how many samples of each class to include
train_samples_per_class = 1000
test_samples_per_class = 200

# Sample the training and testing data
X_train, y_train = sample(X_train, y_train, train_samples_per_class)
X_test, y_test = sample(X_test, y_test, test_samples_per_class)

# Labels come out of cifar10 as (N, 1); flatten so sklearn's metrics accept them.
y_train = y_train.flatten()
y_test  = y_test.flatten()

# Print shapes
print("X_train.shape:", X_train.shape)
print("X_test.shape: ", X_test.shape)

In [ ]:
# Quick look at the labels: 10,000 flat integers, 1,000 of each class.
print(y_train.shape, np.bincount(y_train))
y_train

In [ ]:
# Display a random image, at its native 32x32 resolution
idx = np.random.randint(0, len(X_train))
plt.figure(figsize=(4, 4))
plt.imshow(X_train[idx])
plt.title(f"Label: {cifar_classes[y_train[idx]]}  (32x32 original)")
plt.show()

### 1.2 Resize to the pretrained model's input size

A pretrained network expects the input size it was trained on. EfficientNetV2M wants 224×224, and
CIFAR-10 is 32×32, so every image is upsampled by a factor of seven.

This does not invent detail that was never there — the enlarged images look soft, as the comparison
below shows. What it does is put features at roughly the scale the pretrained filters expect. A
network trained on 224×224 photographs learned filters sized for edges in *those* images; feeding it
32×32 input would present everything at the wrong scale.

> **Memory warning.** `tf.image.resize` returns `float32`, so the training array becomes
> 10,000 × 224 × 224 × 3 × 4 bytes ≈ **6 GB**, plus about 1.2 GB for the test set. If the kernel dies
> here, lower `train_samples_per_class` — or switch to a `tf.data` pipeline that resizes in batches
> rather than materializing everything at once.

In [ ]:
import tensorflow as tf

# Resize the data to match EfficientNetV2M's expected 224x224 input.
# Note: this upcasts to float32, so the arrays get much larger -- see the warning above.
X_train = np.array( tf.image.resize(X_train, (224, 224)) )
X_test = np.array( tf.image.resize(X_test, (224, 224)) )

print("X_train.shape:", X_train.shape)
print("X_test.shape: ", X_test.shape)
print(f"X_train memory: {X_train.nbytes / 1e9:.2f} GB")

In [ ]:
# The same image after resizing -- upsampled to 224x224, and visibly softer.
plt.figure(figsize=(4, 4))
plt.imshow(X_train[idx].astype(int))
plt.title(f"Label: {cifar_classes[y_train[idx]]}  (resized to 224x224)")
plt.show()

## 2. Load the pretrained model

Two arguments do all the work here.

**`weights='imagenet'`** downloads weights trained on 1.2 million labeled photographs — the whole
point of the exercise. Passing `None` instead would give the same architecture with random weights,
which is just training from scratch again.

**`include_top=False`** discards the final classification layers, the part that maps features to
ImageNet's 1,000 specific categories. What remains is the convolutional stack, which outputs a
feature map rather than class probabilities. Those features are general-purpose in a way the
discarded classifier is not.

Then **`base_model.trainable = False`** freezes it. Compare the "Trainable params" and "Non-trainable
params" lines in the summary below: essentially every weight is now non-trainable.

Two model choices are offered. `EfficientNetV2M` is large and accurate; `MobileNetV2` is far
smaller and faster, and worth switching to if the cell below is slow. Either works —
`U2-2_CNN-7_ImageEmbeddings.ipynb` compares what they actually produce.

> **Preprocessing gotcha.** Each pretrained model expects its inputs scaled a particular way.
> EfficientNetV2 models include the rescaling *inside* the network and want raw `[0, 255]` values,
> which is what we have. MobileNetV2 does **not** — it expects inputs mapped to `[-1, 1]` via
> `tensorflow.keras.applications.mobilenet_v2.preprocess_input`. If you switch models, switch the
> preprocessing too, or the borrowed features will be computed on inputs the network never saw
> during its own training.

Keras applications: https://keras.io/api/applications/

In [ ]:
from tensorflow.keras.applications import EfficientNetV2M
from tensorflow.keras.applications import MobileNetV2

# Load the pretrained base WITHOUT its ImageNet classification head.
base_model = EfficientNetV2M(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
#base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze it: these weights will not be updated by training.
base_model.trainable = False

# Display model summary -- check the trainable vs non-trainable parameter counts.
base_model.summary()

## 3. Extract the pretrained features

Because the base is frozen, its output for a given image can never change during training. So rather
than recomputing it every epoch, we run each image through the base **once** and keep the result.

This is a large speedup. Training the head then means running a two-layer network over precomputed
vectors, so fifty epochs cost roughly what a single epoch would through the full base — which is why
we can afford `epochs=50` in section 4.

The output shape is a feature map, not a vector: `(N, 7, 7, 1280)` for EfficientNetV2M. Each of the
1,280 channels is one learned feature detector, and the 7×7 grid records where in the image it fired.
That grid is what 224×224 becomes after the base's repeated downsampling.

> This trick only works while the base is frozen. If you later unfreeze it to fine-tune, the features
> change every step and must be recomputed on the fly — which is why the end-to-end model in
> section 5 exists.

In [ ]:
# Extract features using the pre-trained model
X_train_encoded = base_model.predict(X_train, verbose=1)
X_test_encoded = base_model.predict(X_test, verbose=1)

print("X_train_encoded.shape:", X_train_encoded.shape)
print("X_test_encoded.shape: ", X_test_encoded.shape)

## 4. Train a simple dense head

### 4.1 Build the head

This is the only part being trained, and it is deliberately tiny: `GlobalAveragePooling2D` collapses
the 7×7×1,280 feature map to a 1,280-vector, dropout regularizes, one 32-unit hidden layer mixes the
features, and a softmax produces the ten class scores. A few tens of thousands of parameters against
the base's tens of millions.

Small is the right choice here. The borrowed features are already highly informative, so the head's
only job is to learn which of them matter for *these* ten classes. A large head on 10,000 training
images would mostly overfit.

Note that the head's `Input` is shaped like the **features** (7, 7, 1280), not like an image — it is
being trained on the precomputed arrays from section 3. Section 5 reconnects it to raw pixels.

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *

n_classes  = np.unique(y_train).shape[0]

# Create a classification head
input_encoded = Input(shape=X_train_encoded.shape[1:])
x = GlobalAveragePooling2D()(input_encoded)
x = Dropout(0.25)(x)
x = Dense(32, activation='relu')(x)
outputs = Dense(n_classes, activation='softmax')(x)

# Build the model
model = Model(inputs=input_encoded, outputs=outputs)

model.summary()

### 4.2 Compile and configure training

A learning rate of 0.001 — ten times lower than the from-scratch runs in earlier notebooks. Nothing
here needs to move far: we are fitting a small head on top of good features, not discovering
convolutional filters, so large steps would only overshoot.

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers.schedules import ExponentialDecay, PolynomialDecay

# define training parameters
epochs     = 50
batch_size = 200

optimizer = Adam(
    learning_rate=0.001,
)

# Compile model
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=20,          # Stop after 5 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

### 4.3 Train and evaluate

Compare the result against the from-scratch CNN in `U2-2_CNN-4_Cifar.ipynb` — and note that this run
had only 10,000 training images to that notebook's 50,000, yet should still come out well ahead.
That gap is the value of the borrowed features.

Also watch how few epochs it takes to converge. There are no convolutions to learn, only a small
linear map over features that were already good.

In [ ]:
# Train on the PRECOMPUTED features, not the raw images -- that is what makes this fast.
model, history = helpers.train_and_evaluate(
    model,
    X_train_encoded, y_train,
    X_test_encoded, y_test,
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping],
    class_names=cifar_classes
)

## 5. Stitch the pieces back into one model

### 5.1 Build the combined model

Right now we have two objects: a frozen base that turns images into features, and a trained head that
turns features into labels. Predicting on a new image would mean calling them in sequence by hand.

The functional API joins them into a single model that goes from raw pixels straight to class
probabilities. Both `base_model` and `model` are callable on tensors — a whole model can be used as a
layer — so the wiring is two lines.

`training=False` on the base call keeps its `BatchNormalization` layers in inference mode, using the
statistics learned on ImageNet rather than recomputing them from our batches. Omitting it is a
classic transfer-learning bug: the frozen weights stay frozen, but the batch-norm statistics drift
anyway, and accuracy quietly degrades.

The combined model is what you would save and deploy.

In [ ]:
# Define the new input and pass through base_model
combined_input = Input(shape=(224, 224, 3))

x = base_model(combined_input, training=False)

# Now attach the existing classification head
x = model(x)

# Final combined model
combined_model = Model(inputs=combined_input, outputs=x)
combined_model.summary()

### 5.2 Sanity-check the combined model

A quick end-to-end test: run the first ten training images through the combined model and compare its
predictions against the true labels. They should mostly agree — the model has seen these images, so
disagreement here would mean the wiring in 5.1 is wrong, not that the model generalizes poorly.

This is worth doing every time you assemble a model from parts. A shape mismatch or a wrong
preprocessing step usually shows up immediately as predictions that are no better than chance.

In [ ]:
# Predicted class for the first ten training images, end to end from raw pixels.
np.argmax(combined_model.predict(X_train[:10], verbose=1), axis=1)

In [ ]:
# The true labels for those same ten images -- compare against the row above.
y_train[:10]

In [ ]:
# Spot-check one of them by eye.
plt.imshow(X_train[2].astype(int))
plt.title(f"True: {cifar_classes[y_train[2]]}")
plt.axis('off')
plt.show()

## 6. Review

| | From scratch (`U2-2_CNN-4`) | Transfer learning (this notebook) |
|---|---|---|
| Convolutional weights | Learned from our data | Borrowed from ImageNet, frozen |
| Training images | 50,000 | 10,000 |
| Trainable parameters | All of them | Only the head — a few tens of thousands |
| Input size | 32×32 native | Resized to 224×224 |
| What training does | Discovers filters *and* a classifier | Learns a classifier over fixed features |

**Takeaways**

- **A CNN separates cleanly into features and a classifier.** Only the classifier is task-specific,
  which is exactly why `include_top=False` gives you something reusable. The edge and texture
  detectors ImageNet produced are not about ImageNet's 1,000 categories — they are about images.
- **Freezing is not only an optimization.** It protects the borrowed weights from a randomly
  initialized head, whose early gradients would otherwise be large and meaningless.
- **Precompute frozen features.** A frozen base gives the same output for the same image every time,
  so running it once and training on the results turns fifty epochs into something affordable.
- **Match the input the base expects** — both its size and its preprocessing. A resize to 224×224
  puts features at the scale the filters were trained for, and each Keras application has its own
  expectation about pixel scaling.
- **Assemble at the end, and sanity-check the assembly.** The combined model is what gets deployed;
  `training=False` on the frozen base keeps its batch-norm statistics from drifting, a subtle bug
  that costs accuracy without raising an error.

**Where to take it further:** the natural next step is **fine-tuning** — after the head has trained,
unfreeze the base's last few blocks and continue at a very low learning rate (1e-5 or so), letting
the most task-specific features adapt to CIFAR-10 while the general early layers stay put. Order
matters: fine-tuning before the head has converged undoes the protection freezing gave you.

**Next:** `U2-2_CNN-7_ImageEmbeddings.ipynb` looks at the borrowed feature vectors themselves and
asks how the choice of pretrained model changes them.